# Max tree height (Filtered) maping by random forest models in GEE

In [11]:
# import the libraries
import ee
import pandas as pd
import os
import numpy as np
from termcolor import colored # this is allocate colour and fonts type for the print title and text

In [12]:
#set the working directory of local drive for Grid search result table loading
# os.getcwd()
# os.chdir('D:/Maximum_Height/TreeHeightMap/TreeStructureProject')

In [13]:
# initialize the earth engine API
ee.Initialize(project='ee-sun520ying')

In [14]:
# define the boundary geography reference
unboundedGeo = ee.Geometry.Polygon([-180, 88, 0, 88, 180, 88, 180, -88, 0, -88, -180, -88], None, False)

### Check the data structure of the paramter tables

In [15]:
# define the list of predictors
propertyOfInterest = ['Aridity_Index',
                      'CHELSA_Annual_Mean_Temperature',
                      'CHELSA_Annual_Precipitation',
                      'CHELSA_Isothermality',
                      'CHELSA_Max_Temperature_of_Warmest_Month',
                      'CHELSA_Mean_Diurnal_Range',
                      'CHELSA_Mean_Temperature_of_Coldest_Quarter',
                      'CHELSA_Mean_Temperature_of_Driest_Quarter',
                      'CHELSA_Mean_Temperature_of_Warmest_Quarter',
                      'CHELSA_Mean_Temperature_of_Wettest_Quarter',
                      'CHELSA_Min_Temperature_of_Coldest_Month',
                      'CHELSA_Precipitation_Seasonality',
                      'CHELSA_Precipitation_of_Coldest_Quarter',
                      'CHELSA_Precipitation_of_Driest_Month',
                      'CHELSA_Precipitation_of_Driest_Quarter',
                      'CHELSA_Precipitation_of_Warmest_Quarter',
                      'CHELSA_Precipitation_of_Wettest_Month',
                      'CHELSA_Precipitation_of_Wettest_Quarter',
                      'CHELSA_Temperature_Annual_Range',
                      'CHELSA_Temperature_Seasonality',
                      'Depth_to_Water_Table',
                      'EarthEnvTopoMed_Eastness',
                      'EarthEnvTopoMed_Elevation',
                      'EarthEnvTopoMed_Northness',
                      'EarthEnvTopoMed_ProfileCurvature',
                      'EarthEnvTopoMed_Roughness',
                      'EarthEnvTopoMed_Slope',
                      'SG_Absolute_depth_to_bedrock',
                      'WorldClim2_SolarRadiation_AnnualMean',
                      'WorldClim2_WindSpeed_AnnualMean',
                      'EarthEnvCloudCover_MODCF_interannualSD',
                      'EarthEnvCloudCover_MODCF_intraannualSD',
                      'EarthEnvCloudCover_MODCF_meanannual',
                      'EarthEnvTopoMed_AspectCosine',
                      'EarthEnvTopoMed_AspectSine',
                      'LandCoverClass_Cultivated_and_Managed_Vegetation',
                      'Human_Disturbance',
                      'LandCoverClass_Urban_Builtup',
                      'SG_Clay_Content_0_100cm',
                      'SG_Coarse_fragments_0_100cm',
                      'SG_Sand_Content_0_100cm',
                      'SG_Silt_Content_0_100cm',
                      'SG_Soil_pH_H2O_0_100cm',
                      'WDPA',
                      'cropland',
                      'grazing',
                      'pasture',
                      'rangeland',
                      'PresentTreeCover']
print(propertyOfInterest)

['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Northness', 'EarthEnvTopoMed_ProfileCurvature', 'EarthEnvTopoMed_Roughness', 'EarthEnvTopoMed_Slope', 'SG_Absolute_depth_to_bedrock', 'World

In [16]:
# read the composite
compositeImage = ee.Image("users/leonidmoore/ForestBiomass/20200915_Forest_Biomass_Predictors_Image").select(propertyOfInterest)
# show the band names of the composite image 
print('Composite Band Names:',compositeImage.bandNames().getInfo())

Composite Band Names: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Northness', 'EarthEnvTopoMed_ProfileCurvature', 'EarthEnvTopoMed_Roughness', 'EarthEnvTopoMed_Slope', 'SG_Absolute_dep

In [17]:
# toggle these human activity layers into zero activity situation
toggledCultivated = compositeImage.select('LandCoverClass_Cultivated_and_Managed_Vegetation').lt(0)
toggledUrban = compositeImage.select('LandCoverClass_Urban_Builtup').lt(0)
toggledDisturbance = compositeImage.select('Human_Disturbance').lt(0)
toggledCropland = compositeImage.select('cropland').lt(0)
toggledGrazing = compositeImage.select('grazing').lt(0)
toggledPasture = compositeImage.select('pasture').lt(0)
toggledRangeland = compositeImage.select('rangeland').lt(0)
toggledWDPA = compositeImage.select('WDPA').gte(0)
# load the potential tree cover and rename it to 'PresentTreeCover'
potentialTreeCover = ee.Image('users/leonidmoore/ForestBiomass/Bastin_et_al_2019_Potential_Forest_Cover_Adjusted').rename("PresentTreeCover")

In [18]:
# define the list of retained predictors
retainedPropeties = ['Aridity_Index',
                      'CHELSA_Annual_Mean_Temperature',
                      'CHELSA_Annual_Precipitation',
                      'CHELSA_Isothermality',
                      'CHELSA_Max_Temperature_of_Warmest_Month',
                      'CHELSA_Mean_Diurnal_Range',
                      'CHELSA_Mean_Temperature_of_Coldest_Quarter',
                      'CHELSA_Mean_Temperature_of_Driest_Quarter',
                      'CHELSA_Mean_Temperature_of_Warmest_Quarter',
                      'CHELSA_Mean_Temperature_of_Wettest_Quarter',
                      'CHELSA_Min_Temperature_of_Coldest_Month',
                      'CHELSA_Precipitation_Seasonality',
                      'CHELSA_Precipitation_of_Coldest_Quarter',
                      'CHELSA_Precipitation_of_Driest_Month',
                      'CHELSA_Precipitation_of_Driest_Quarter',
                      'CHELSA_Precipitation_of_Warmest_Quarter',
                      'CHELSA_Precipitation_of_Wettest_Month',
                      'CHELSA_Precipitation_of_Wettest_Quarter',
                      'CHELSA_Temperature_Annual_Range',
                      'CHELSA_Temperature_Seasonality',
                      'Depth_to_Water_Table',
                      'EarthEnvTopoMed_Eastness',
                      'EarthEnvTopoMed_Elevation',
                      'EarthEnvTopoMed_Northness',
                      'EarthEnvTopoMed_ProfileCurvature',
                      'EarthEnvTopoMed_Roughness',
                      'EarthEnvTopoMed_Slope',
                      'SG_Absolute_depth_to_bedrock',
                      'WorldClim2_SolarRadiation_AnnualMean',
                      'WorldClim2_WindSpeed_AnnualMean',
                      'EarthEnvCloudCover_MODCF_interannualSD',
                      'EarthEnvCloudCover_MODCF_intraannualSD',
                      'EarthEnvCloudCover_MODCF_meanannual',
                      'EarthEnvTopoMed_AspectCosine',
                      'EarthEnvTopoMed_AspectSine',
                      'SG_Clay_Content_0_100cm',
                      'SG_Coarse_fragments_0_100cm',
                      'SG_Sand_Content_0_100cm',
                      'SG_Silt_Content_0_100cm',
                      'SG_Soil_pH_H2O_0_100cm']
print(retainedPropeties[0:5])


['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month']


In [19]:
# replace the human activity layers in the compositeImageRaw
compositeImageUpdated = compositeImage.select(retainedPropeties).addBands(toggledCultivated).addBands(toggledUrban).addBands(toggledDisturbance).addBands(toggledCropland).addBands(toggledGrazing).addBands(toggledPasture).addBands(toggledRangeland).addBands(toggledWDPA).addBands(potentialTreeCover)
# present the composite band names
print(colored('The band names are:', 'blue', attrs=['bold']),compositeImage.bandNames().getInfo())

The band names are: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Northness', 'EarthEnvTopoMed_ProfileCurvature', 'EarthEnvTopoMed_Roughness', 'EarthEnvTopoMed_Slope', 'SG_Absolute_depth

In [20]:
predictorsSelected = compositeImageUpdated.bandNames()

In [23]:
# load the grid searh table from R
parameterTable = pd.read_csv('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject/ModelOptimization/GridSearchResultsGEE/PotentialTree_Height_filtered_Buffer_Based_20260407_Subsample_Grid_Search_Seed_0.csv', float_precision='round_trip')
# show the structure by head function
print(colored('The head of the table: \n', 'blue', attrs=['bold']))
parameterTable.head()

The head of the table: 



,Unnamed: 0,Mean_R2,ModelName,StDev_R2,type,numberOfTrees,variablesPerSplit,minLeafPopulation,bagFraction,maxNodes
0,0,0.427887,GridSeach_Model_15_2_90,0.075561,Classifier.smileRandomForest,500,15,2,0.632,90


### After you run the chunk below, please check the status of the maping tasks on your Google earth engine

In [26]:
# define a loop through the seed list
seedList = np.arange(0, 50, 1).tolist()
print(colored('The models are:', 'blue', attrs=['bold']),seedList)
print(colored('Model is running:\nWith paramter sets:', 'blue', attrs=['bold']))
# for seed in seedList: range(0,len(seedList))
for seed in seedList:
    # load the points data in shape file format
    woodDensityPoints = ee.FeatureCollection('users/sun520ying/TreeStructureProject/TrainTables/TreeHeight_filtered_BufferZone_20260407_Subsampled_Train_table_Seed_'+str(seed))
    # check the information of the FeatureCollection
    # print(woodDensityTable.first().getInfo())
    # extract the environment covariates
    extractedVariableTable = compositeImageUpdated.select(predictorsSelected).reduceRegions(collection = woodDensityPoints,
                                                                                     reducer = ee.Reducer.first())
    # show the str of the extracted data
    # print(extractedVariableTable.first().getInfo())
    # exclude the rows with null valus
    trainTable = extractedVariableTable.filter(ee.Filter.notNull(predictorsSelected))
    # print(trainTable.size().getInfo())
    parameterTable = pd.read_csv('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject/ModelOptimization/GridSearchResultsGEE/PotentialTree_Height_filtered_Buffer_Based_20260407_Subsample_Grid_Search_Seed_'+str(seed)+'.csv', float_precision='round_trip')
    # not recomend to run the code below
    # print(parameterTable.head())
    # extract the paramters
    variablesPerSplitVal = int(parameterTable['variablesPerSplit'].iat[0]) # mtry
    minLeafPopulationVal = int(parameterTable['minLeafPopulation'].iat[0]) # minrow
    maxNodesVal = int(parameterTable['maxNodes'].iat[0]) # mac depth
    print('seed',seed,variablesPerSplitVal,minLeafPopulationVal,maxNodesVal)
    # define the random forest classifier
    rfClassifier = ee.Classifier.smileRandomForest(numberOfTrees = 500,
                                                   variablesPerSplit = variablesPerSplitVal, # mtry
                                                   minLeafPopulation = minLeafPopulationVal, # minrow
                                                   maxNodes = maxNodesVal, # max depth
                                                   bagFraction = 0.632,
                                                   seed = seed).setOutputMode('REGRESSION')
    trainedClassifier = rfClassifier.train(features = trainTable,
                                           classProperty = 'MaxHeight',
                                           inputProperties = predictorsSelected)
    # execute the prediction to generate the map
    predictedWoodDensityMap = compositeImageUpdated.classify(trainedClassifier)
    # print(predictedWoodDensityMap.getInfo())
    predictionExport = ee.batch.Export.image.toAsset(image = predictedWoodDensityMap,
                                                     description = '20260407_TreeHeight_Map_To_Asset_'+str(seed),
                                                     assetId = 'users/sun520ying/TreeStructureProject/PredictedMaps/Predicted_PotentialMax_TreeHeight_Filtered_BufferBased_20260407_Mean_Map_with_Seed_'+str(seed),
                                                     region = unboundedGeo,
                                                     crs = 'EPSG:4326',
                                                     crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                                     maxPixels = 1e13)

    # print(predictionExportAsset)
    # start the export task
    predictionExport.start()
    # show the task status
    predictionExport.status()

The models are: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Model is running:
With paramter sets:
seed 0 15 2 90
seed 1 18 2 80
seed 2 9 4 80
seed 3 12 2 90
seed 4 6 2 90
seed 5 15 4 80
seed 6 15 4 100
seed 7 18 4 100
seed 8 18 2 100
seed 9 9 2 100
seed 10 21 4 80
seed 11 21 2 100
seed 12 21 4 60
seed 13 9 4 100
seed 14 9 2 100
seed 15 6 4 100
seed 16 12 2 100
seed 17 15 4 100
seed 18 18 4 100
seed 19 21 2 100
seed 20 15 4 70
seed 21 6 4 90
seed 22 12 4 100
seed 23 21 2 70
seed 24 12 4 90
seed 25 9 4 100
seed 26 15 4 90
seed 27 12 2 100
seed 28 9 4 90
seed 29 18 2 100
seed 30 15 2 70
seed 31 9 2 100
seed 32 9 4 90
seed 33 12 2 90
seed 34 9 4 80
seed 35 18 2 90
seed 36 9 2 90
seed 37 9 4 90
seed 38 21 2 50
seed 39 12 2 70
seed 40 21 4 100
seed 41 6 6 90
seed 42 21 4 90
seed 43 6 2 100
seed 44 15 4 90
seed 45 21 2 80
seed 46 21 2 100
seed 47 3